In [ ]:
import os
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import SimpleITK as sitk
import pandas as pd
import torch
import torch.nn.functional as F
import math

In [ ]:
def plot_image(img, seg):
    """
    img: H x W x 3
    seg: H x W
    """

    assert img.shape[:2] == seg.shape

    fig = go.Figure()

    # Base RGB image (trace 0)
    fig.add_trace(
        go.Image(z=img)
    )

    # Overlay (trace 1)
    fig.add_trace(
        go.Heatmap(
            z=seg,
            colorscale="Jet",
            opacity=0.4,
            zmin=seg.min(),
            zmax=seg.max(),
            showscale=True,
            hoverinfo="skip"
        )
    )

    # ---- Opacity slider ----
    opacities = np.linspace(0, 1, 11)

    steps = [
        dict(
            method="restyle",
            args=[{"opacity": [None, o]}, [0, 1]],
            label=f"{o:.1f}"
        )
        for o in opacities
    ]

    fig.update_layout(
        sliders=[dict(
            active=4,
            currentvalue={"prefix": "Overlay opacity: "},
            pad={"t": 30},
            steps=steps
        )],
        xaxis=dict(
            showticklabels=False,
            range=[0, img.shape[1]]
        ),
        yaxis=dict(
            showticklabels=False,
            range=[img.shape[0], 0],
            scaleanchor="x"
        ),
        margin=dict(l=0, r=0, t=20, b=0),
    )

    fig.show()

In [ ]:
mount_point = '/CMF/data/lumargot/trachoma/'
img = 'B images one eye/img/1183_B-RIGHT.jpg'
seg = 'B images one eye/seg/1183_B-RIGHT.nrrd'

In [ ]:
img_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, img)))
seg_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, seg)))

In [ ]:
plot_image(img_np, seg_np)

In [ ]:
df_train = pd.read_csv('/CMF/data/lumargot/trachoma/csv_mtss_pret/csv_updated/mtss_pret_combined_train.csv')


In [ ]:
df_train

In [ ]:
mount_point = '/CMF/data/lumargot/trachoma/'


In [ ]:
img_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, 'PoPP_Data/besrat/img/1543_blsx_os_postop.jpg')))
seg_np = sitk.GetArrayFromImage(sitk.ReadImage(os.path.join(mount_point, 'PoPP_Data/besrat/seg/1543_blsx_os_postop.nrrd')))

In [ ]:
def _resize_with_grid_sample(
    x: torch.Tensor,
    out_hw: tuple[int, int],
    mode: str,
    align_corners: bool = False,
) -> torch.Tensor:
    """
    x: (N,C,H,W) tensor
    out_hw: (H_out, W_out)
    mode: 'bilinear' for images/logits, 'nearest' for label maps
    """
    assert x.ndim == 4, f"Expected (N,C,H,W), got {x.shape}"

    mesh_grid_params = [torch.arange(start=-1.0, end=1.0, step=(2.0/s), device=x.device) for s in out_hw]
    mesh_grid = torch.stack(torch.meshgrid(mesh_grid_params, indexing='xy'), dim=-1).to(torch.float32).unsqueeze(0)

    y = F.grid_sample(
        x, mesh_grid,
        mode=mode,
        padding_mode="zeros",
        align_corners=align_corners,
    )
    return y

In [ ]:
img_np_512 = _resize_with_grid_sample(torch.from_numpy(img_np).permute(2,0,1).unsqueeze(0).float(), (512, 512), mode='bilinear').squeeze(0).permute(1,2,0).numpy()
seg_np_512 = _resize_with_grid_sample(torch.from_numpy(seg_np).unsqueeze(0).unsqueeze(0).float(), (512, 512), mode='nearest').squeeze(0).squeeze(0).numpy()

plot_image(img_np_512, seg_np_512)

In [ ]:
import torch
import torch.nn.functional as F
from typing import Dict, Sequence, Tuple, Optional

class RandZoomdTorch:
    """
    RandZoomd-like transform for dict batches.
    Applies the SAME random zoom (and optional center jitter) to all tensors in `keys`.

    Expected each entry to be (B,C,H,W) or (C,H,W). If (C,H,W), it will be treated as B=1.
    """
    def __init__(
        self,
        keys: Sequence[str],
        zoom_range: Tuple[float, float] = (0.8, 1.2),
        prob: float = 1.0,
        center_jitter: float = 0.0,
        padding_mode: str = "zeros",
        align_corners: bool = False,
        mode_map: Optional[Dict[str, str]] = None,  # e.g. {"img": "bilinear", "seg": "nearest"}
    ):
        self.keys = list(keys)
        self.zoom_range = zoom_range
        self.prob = prob
        self.center_jitter = center_jitter
        self.padding_mode = padding_mode
        self.align_corners = align_corners
        self.mode_map = mode_map or {}

    @torch.no_grad()
    def _make_grid(self, B: int, H: int, W: int, device, dtype):
        ys = torch.linspace(-1.0, 1.0, H, device=device, dtype=dtype)
        xs = torch.linspace(-1.0, 1.0, W, device=device, dtype=dtype)
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        base = torch.stack([gx, gy], dim=-1)  # (H,W,2)
        return base.unsqueeze(0).expand(B, -1, -1, -1).contiguous()

    def __call__(self, data: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        out = dict(data)

        # Find a reference tensor to define B,H,W + device/dtype
        ref = None
        for k in self.keys:
            if k in out:
                ref = out[k]
                break
        if ref is None:
            return out

        # Normalize shapes to (B,C,H,W)
        def ensure_bchw(x: torch.Tensor):
            if x.ndim == 3:
                return x.unsqueeze(0), True  # added batch
            if x.ndim == 4:
                return x, False
            raise ValueError(f"Expected (C,H,W) or (B,C,H,W), got {tuple(x.shape)}")

        ref_bchw, ref_added_batch = ensure_bchw(ref)
        B, _, H, W = ref_bchw.shape
        device, dtype = ref_bchw.device, ref_bchw.dtype

        # Single random draw for this call -> SAME transform for all keys
        if torch.rand(1, device=device).item() >= self.prob:
            return out  # no-op

        zmin, zmax = self.zoom_range
        if zmin > zmax:
            zmin, zmax = zmax, zmin
        zoom = torch.empty(1, device=device, dtype=dtype).uniform_(zmin, zmax).item()

        if self.center_jitter > 0:
            cx = torch.empty(1, device=device, dtype=dtype).uniform_(-self.center_jitter, self.center_jitter).item()
            cy = torch.empty(1, device=device, dtype=dtype).uniform_(-self.center_jitter, self.center_jitter).item()
        else:
            cx, cy = 0.0, 0.0

        # Build one grid and reuse for all keys
        base_grid = self._make_grid(B, H, W, device=device, dtype=dtype)
        c = torch.tensor([cx, cy], device=device, dtype=dtype).view(1, 1, 1, 2)
        grid = (base_grid - c) / zoom + c  # (B,H,W,2)

        # Apply to each key with per-key interpolation mode (img vs seg)
        for k in self.keys:
            if k not in out:
                continue
            x, added_batch = ensure_bchw(out[k])

            # Sanity: require same spatial size; if not, you can resample per-key separately
            if x.shape[-2:] != (H, W):
                raise ValueError(
                    f"All keys must share same H,W. Key '{k}' has {tuple(x.shape[-2:])}, ref is {(H,W)}."
                )

            mode = self.mode_map.get(k, "bilinear")
            y = F.grid_sample(
                x,
                grid,
                mode=mode,
                padding_mode=self.padding_mode,
                align_corners=self.align_corners,
            )

            out[k] = y.squeeze(0) if added_batch else y

        return out


In [ ]:
px.imshow(img_np)

In [ ]:
transform = RandZoomdTorch(
    keys=['img', 'seg'],
    zoom_range=(0.2, 1.5),
    prob=1.0,
    center_jitter=0.1,
    padding_mode="zeros",
    align_corners=False,
    mode_map={"img": "bilinear", "seg": "nearest"},
)
data = {
    'img': torch.from_numpy(img_np).permute(2,0,1).unsqueeze(0).float(),  # (1,C,H,W)
    'seg': torch.from_numpy(seg_np).unsqueeze(0).unsqueeze(0).float(),      # (1,1,H,W)
}


In [ ]:
out = transform(data)   


In [ ]:
import torch
import torch.nn.functional as F
from typing import Dict, Sequence, Tuple, Optional

class ResizeGridSampledTorch:
    """
    Dict-style resize using grid_sample.
    Applies the SAME resize mapping to all tensors in `keys`.

    Input tensors must be (B,C,H,W) or (C,H,W). All keys must share the same H,W.
    """
    def __init__(
        self,
        keys: Sequence[str],
        out_size: Tuple[int, int],                 # (H_out, W_out)
        padding_mode: str = "zeros",
        align_corners: bool = False,
        mode_map: Optional[Dict[str, str]] = None, # {"img":"bilinear","seg":"nearest"}
    ):
        self.keys = list(keys)
        self.out_size = tuple(out_size)
        self.padding_mode = padding_mode
        self.align_corners = align_corners
        self.mode_map = mode_map or {}

    @staticmethod
    def _ensure_bchw(x: torch.Tensor):
        if x.ndim == 3:
            return x.unsqueeze(0), True
        if x.ndim == 4:
            return x, False
        raise ValueError(f"Expected (C,H,W) or (B,C,H,W), got {tuple(x.shape)}")

    def _make_resize_grid(
        self,
        B: int,
        H_in: int,
        W_in: int,
        H_out: int,
        W_out: int,
        device,
        dtype,
    ) -> torch.Tensor:
        """
        Builds grid (B, H_out, W_out, 2) in normalized coords [-1,1],
        mapping output pixel locations to input sampling locations.
        """
        # Output coordinates in normalized space of output grid
        ys = torch.linspace(-1.0, 1.0, H_out, device=device, dtype=dtype)
        xs = torch.linspace(-1.0, 1.0, W_out, device=device, dtype=dtype)
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")  # (H_out, W_out)

        # For pure resize, we can sample input at the same normalized coords.
        # This is correct because grid_sample interprets normalized coords on the *input* domain.
        grid = torch.stack([gx, gy], dim=-1)  # (H_out, W_out, 2)
        return grid.unsqueeze(0).expand(B, -1, -1, -1).contiguous()

    def __call__(self, data: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        out = dict(data)

        # pick reference tensor to define B,H,W + device/dtype
        ref = None
        for k in self.keys:
            if k in out:
                ref = out[k]
                break
        if ref is None:
            return out

        ref, ref_added_batch = self._ensure_bchw(ref)
        B, _, H_in, W_in = ref.shape
        H_out, W_out = self.out_size
        device, dtype = ref.device, ref.dtype

        grid = self._make_resize_grid(B, H_in, W_in, H_out, W_out, device, dtype)

        # Apply same grid to all keys
        for k in self.keys:
            if k not in out:
                continue
            x, added_batch = self._ensure_bchw(out[k])

            if x.shape[-2:] != (H_in, W_in):
                raise ValueError(
                    f"All keys must share same input H,W. Key '{k}' has {tuple(x.shape[-2:])}, ref is {(H_in,W_in)}."
                )

            mode = self.mode_map.get(k, "bilinear")
            y = F.grid_sample(
                x,
                grid,
                mode=mode,
                padding_mode=self.padding_mode,
                align_corners=self.align_corners,
            )
            out[k] = y.squeeze(0) if added_batch else y

        return out


In [ ]:
transform_resize = ResizeGridSampledTorch(
    keys=['img', 'seg'],
    out_size=(512, 512),
    padding_mode="zeros",
    align_corners=False,
    mode_map={"img": "bilinear", "seg": "nearest"},
)

data = {
    'img': torch.from_numpy(img_np).permute(2,0,1).unsqueeze(0).float(),  # (1,C,H,W)
    'seg': torch.from_numpy(seg_np).unsqueeze(0).unsqueeze(0).float(),      # (1,1,H,W)
}


In [ ]:
out_res = transform_resize(out)

In [ ]:
plot_image(out_res['img'].squeeze(0).permute(1,2,0).numpy(), out_res['seg'].squeeze(0).squeeze(0).numpy())

In [ ]:
class RandRotateGridSampledTorch:
    """
    Dict-style random rotation using grid_sample.
    Applies the SAME random rotation to all tensors in `keys`.

    Supports tensors shaped (B,C,H,W) or (C,H,W).
    All keys must share the same H,W.
    """
    def __init__(
        self,
        keys: Sequence[str],
        angle_range: Tuple[float, float] = (-15.0, 15.0),  # degrees by default
        prob: float = 1.0,
        degrees: bool = True,
        center: str = "image",              # "image" only in this simple version
        translate: Tuple[float, float] = (0.0, 0.0),  # max translation in normalized coords (tx, ty)
        padding_mode: str = "zeros",
        align_corners: bool = False,
        mode_map: Optional[Dict[str, str]] = None,      # {"img":"bilinear","seg":"nearest"}
    ):
        self.keys = list(keys)
        self.angle_range = angle_range
        self.prob = prob
        self.degrees = degrees
        self.center = center
        self.translate = translate
        self.padding_mode = padding_mode
        self.align_corners = align_corners
        self.mode_map = mode_map or {}

    @staticmethod
    def _ensure_bchw(x: torch.Tensor):
        if x.ndim == 3:
            return x.unsqueeze(0), True
        if x.ndim == 4:
            return x, False
        raise ValueError(f"Expected (C,H,W) or (B,C,H,W), got {tuple(x.shape)}")

    def _base_grid(self, B: int, H: int, W: int, device, dtype):
        ys = torch.linspace(-1.0, 1.0, H, device=device, dtype=dtype)
        xs = torch.linspace(-1.0, 1.0, W, device=device, dtype=dtype)
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        base = torch.stack([gx, gy], dim=-1)  # (H,W,2)
        return base.unsqueeze(0).expand(B, -1, -1, -1).contiguous()

    def __call__(self, data: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        out = dict(data)

        # Reference tensor
        ref = None
        for k in self.keys:
            if k in out:
                ref = out[k]
                break
        if ref is None:
            return out

        ref, _ = self._ensure_bchw(ref)
        B, _, H, W = ref.shape
        device, dtype = ref.device, ref.dtype

        # Decide apply once per call (same params for all keys)
        if torch.rand(1, device=device).item() >= self.prob:
            return out

        a0, a1 = self.angle_range
        if a0 > a1:
            a0, a1 = a1, a0

        angle = torch.empty(1, device=device, dtype=dtype).uniform_(a0, a1).item()
        if self.degrees:
            angle = math.radians(angle)

        # Optional translation (in normalized coords)
        tx_max, ty_max = self.translate
        if tx_max > 0:
            tx = torch.empty(1, device=device, dtype=dtype).uniform_(-tx_max, tx_max).item()
        else:
            tx = 0.0
        if ty_max > 0:
            ty = torch.empty(1, device=device, dtype=dtype).uniform_(-ty_max, ty_max).item()
        else:
            ty = 0.0

        cos_a = math.cos(angle)
        sin_a = math.sin(angle)

        # Build a single grid: rotate around image center in normalized coords
        base = self._base_grid(B, H, W, device=device, dtype=dtype)  # (B,H,W,2)
        x = base[..., 0]
        y = base[..., 1]

        # Rotation about (0,0) is rotation about image center because normalized coords are centered.
        x2 = cos_a * x - sin_a * y + tx
        y2 = sin_a * x + cos_a * y + ty

        grid = torch.stack([x2, y2], dim=-1)  # (B,H,W,2)

        # Apply to all keys with per-key interpolation
        for k in self.keys:
            if k not in out:
                continue
            t, added_batch = self._ensure_bchw(out[k])

            if t.shape[-2:] != (H, W):
                raise ValueError(
                    f"All keys must share same H,W. Key '{k}' has {tuple(t.shape[-2:])}, ref is {(H,W)}."
                )

            mode = self.mode_map.get(k, "bilinear")
            yk = F.grid_sample(
                t,
                grid,
                mode=mode,
                padding_mode=self.padding_mode,
                align_corners=self.align_corners,
            )
            out[k] = yk.squeeze(0) if added_batch else yk

        return out
    


In [ ]:
transform_rot = RandRotateGridSampledTorch(keys=["img", "seg"], angle_range=(-90, 90), prob=0.8, degrees=True, translate=(0.0, 0.0), mode_map={"img": "bilinear", "seg": "nearest"}, padding_mode="border")
out_rot = transform_rot(out_res)

In [ ]:
plot_image(out_rot['img'].squeeze(0).permute(1,2,0).numpy(), out_rot['seg'].squeeze(0).squeeze(0).numpy())